In [9]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# DATA LOADING & PREPROCESSING
train_df = pd.read_csv("train.csv", index_col=0)
test_df = pd.read_csv("test.csv", index_col=0)

# Drop id, date, zipcode
drop_cols_train = [c for c in ["id", "date", "zipcode"] if c in train_df.columns]
drop_cols_test = [c for c in ["id", "date", "zipcode"] if c in test_df.columns]
train_df = train_df.drop(columns=drop_cols_train)
test_df = test_df.drop(columns=drop_cols_test)

# Separate features and target
y_train = train_df["price"].values / 1000.0
y_test = test_df["price"].values / 1000.0
X_train = train_df.drop(columns=["price"]).values.astype(float)
X_test = test_df.drop(columns=["price"]).values.astype(float)

feature_names = train_df.drop(columns=["price"]).columns.tolist()

# Standardize features (mean=0, std=1) using training set stats
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("Features:", feature_names)
print(f"Train: {X_train_sc.shape}, Test: {X_test_sc.shape}")


Features: ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'lat', 'long', 'sqft_living15', 'sqft_lot15']
Train: (1000, 17), Test: (1000, 17)


In [27]:
# PROBLEM 2: Linear Regression with sklearn
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

print("\nCoefficients (θ₁...θ_d):")
for name, coef in zip(feature_names, lr.coef_):
    print(f"  {name:20s}: {coef:.4f}")
print(f"  {'Intercept':20s}: {lr.intercept_:.4f}")

# Part 1: Training metrics
y_train_pred = lr.predict(X_train_sc)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
print(f"\nTraining MSE:  {train_mse:.4f}")
print(f"Training R²:   {train_r2:.4f}")

# Part 2: Testing metrics
y_test_pred = lr.predict(X_test_sc)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)
print(f"\nTesting MSE:   {test_mse:.4f}")
print(f"Testing R²:    {test_r2:.4f}")



Coefficients (θ₁...θ_d):
  bedrooms            : -12.5220
  bathrooms           : 18.5276
  sqft_living         : 56.7488
  sqft_lot            : 10.8819
  floors              : 8.0437
  waterfront          : 63.7429
  view                : 48.2001
  condition           : 12.9643
  grade               : 92.2315
  sqft_above          : 48.2901
  sqft_basement       : 27.1370
  yr_built            : -67.6431
  yr_renovated        : 17.2714
  lat                 : 78.3757
  long                : -1.0352
  sqft_living15       : 45.5777
  sqft_lot15          : -12.9301
  Intercept           : 520.4148

Training MSE:  31486.1678
Training R²:   0.7265

Testing MSE:   57628.1547
Testing R²:    0.6544


In [16]:
# PROBLEM 3: Closed-form solution
def closed_form_fit(X, y):
    """Closed-form: theta = (X^T X)^{-1} X^T y (with bias column)."""
    N = X.shape[0]
    X_b = np.hstack([np.ones((N, 1)), X])  # add bias column
    theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
    return theta  # theta[0] = intercept, theta[1:] = coefficients

def closed_form_predict(X, theta):
    N = X.shape[0]
    X_b = np.hstack([np.ones((N, 1)), X])
    return X_b @ theta

theta_cf = closed_form_fit(X_train_sc, y_train)
print(f"\nIntercept: {theta_cf[0]:.4f}")
print("Coefficients:")
for name, coef in zip(feature_names, theta_cf[1:]):
    print(f"  {name:20s}: {coef:.4f}")

# Training metrics
y_train_cf = closed_form_predict(X_train_sc, theta_cf)
cf_train_mse = mean_squared_error(y_train, y_train_cf)
cf_train_r2 = r2_score(y_train, y_train_cf)
print(f"\nTraining MSE:  {cf_train_mse:.4f}")
print(f"Training R²:   {cf_train_r2:.4f}")

# Testing metrics
y_test_cf = closed_form_predict(X_test_sc, theta_cf)
cf_test_mse = mean_squared_error(y_test, y_test_cf)
cf_test_r2 = r2_score(y_test, y_test_cf)
print(f"\nTesting MSE:   {cf_test_mse:.4f}")
print(f"Testing R²:    {cf_test_r2:.4f}")

# Compare with sklearn
print("\n--- Comparison with sklearn ---")
print(f"Intercept diff:    {abs(theta_cf[0] - lr.intercept_):.10f}")
print(f"Max coef diff:     {np.max(np.abs(theta_cf[1:] - lr.coef_)):.10f}")
print(f"Train MSE diff:    {abs(cf_train_mse - train_mse):.10f}")
print(f"Test MSE diff:     {abs(cf_test_mse - test_mse):.10f}")



Intercept: 520.4148
Coefficients:
  bedrooms            : -33.6896
  bathrooms           : -15.3553
  sqft_living         : 40.0107
  sqft_lot            : 8.9222
  floors              : -0.9664
  waterfront          : 63.9582
  view                : 48.9520
  condition           : 14.1036
  grade               : 89.4960
  sqft_above          : 140.3917
  sqft_basement       : 65.4396
  yr_built            : -67.6431
  yr_renovated        : 17.2714
  lat                 : 78.3757
  long                : -1.0352
  sqft_living15       : 45.5777
  sqft_lot15          : -12.9301

Training MSE:  34144.1954
Training R²:   0.7034

Testing MSE:   56177.3402
Testing R²:    0.6631

--- Comparison with sklearn ---
Intercept diff:    0.0000000000
Max coef diff:     92.1016076552
Train MSE diff:    2658.0275861941
Test MSE diff:     1450.8145130022


In [19]:
# PROBLEM 4: Polynomial Regression

# Use raw sqft_living, then create polynomial features and scale
sqft_idx = feature_names.index("sqft_living")
X_sqft_train = train_df["sqft_living"].values.reshape(-1, 1).astype(float)
X_sqft_test = test_df["sqft_living"].values.reshape(-1, 1).astype(float)

print(f"\n{'p':>3} | {'Train MSE':>12} {'Train R²':>10} | {'Test MSE':>12} {'Test R²':>10}")
print("-" * 65)

for p in range(1, 6):
    # Create polynomial features [X, X^2, ..., X^p]
    X_poly_train = np.hstack([X_sqft_train ** i for i in range(1, p + 1)])
    X_poly_test = np.hstack([X_sqft_test ** i for i in range(1, p + 1)])

    # Scale polynomial features
    poly_scaler = StandardScaler()
    X_poly_train_sc = poly_scaler.fit_transform(X_poly_train)
    X_poly_test_sc = poly_scaler.transform(X_poly_test)

    # Fit using closed-form
    theta_poly = closed_form_fit(X_poly_train_sc, y_train)
    y_poly_train_pred = closed_form_predict(X_poly_train_sc, theta_poly)
    y_poly_test_pred = closed_form_predict(X_poly_test_sc, theta_poly)

    tr_mse = mean_squared_error(y_train, y_poly_train_pred)
    tr_r2 = r2_score(y_train, y_poly_train_pred)
    te_mse = mean_squared_error(y_test, y_poly_test_pred)
    te_r2 = r2_score(y_test, y_poly_test_pred)

    print(f"{p:3d} | {tr_mse:12.4f} {tr_r2:10.4f} | {te_mse:12.4f} {te_r2:10.4f}")



  p |    Train MSE   Train R² |     Test MSE    Test R²
-----------------------------------------------------------------
  1 |   57947.5262     0.4967 |   88575.9785     0.4687
  2 |   54822.6651     0.5238 |   71791.6795     0.5694
  3 |   53785.1947     0.5329 |   99833.4838     0.4012
  4 |   52795.7748     0.5415 |  250979.2743    -0.5053
  5 |   52626.1120     0.5429 |  570616.9148    -2.4225


In [26]:
# PROBLEM 5: Gradient Descent
def gradient_descent(X, y, alpha, n_iters):
    """Gradient descent for linear regression with bias term."""
    N, d = X.shape
    X_b = np.hstack([np.ones((N, 1)), X])
    theta = np.zeros(d + 1)

    for _ in range(n_iters):
        predictions = X_b @ theta
        errors = predictions - y
        gradient = (2.0 / N) * (X_b.T @ errors)
        theta = theta - alpha * gradient

    return theta

alphas = [0.01, 0.1, 0.5]
iterations = [10, 50, 100]

print(f"\n{'α':>6} | {'Iters':>5} | {'Train MSE':>12} {'Train R²':>10} | {'Test MSE':>12} {'Test R²':>10}")
print("-" * 75)

for alpha in alphas:
    for n_iter in iterations:
        theta_gd = gradient_descent(X_train_sc, y_train, alpha, n_iter)
        y_tr_gd = closed_form_predict(X_train_sc, theta_gd)
        y_te_gd = closed_form_predict(X_test_sc, theta_gd)

        gd_tr_mse = mean_squared_error(y_train, y_tr_gd)
        gd_tr_r2 = r2_score(y_train, y_tr_gd)
        gd_te_mse = mean_squared_error(y_test, y_te_gd)
        gd_te_r2 = r2_score(y_test, y_te_gd)

        if gd_tr_mse > 1e10:
            print(f"{alpha:6.2f} | {n_iter:5d} | {'DIVERGED':>12} {'':>10} | {'DIVERGED':>12} {'':>10}")
        else:
            print(f"{alpha:6.2f} | {n_iter:5d} | {gd_tr_mse:12.4f} {gd_tr_r2:10.4f} | {gd_te_mse:12.4f} {gd_te_r2:10.4f}")
# Show final theta for best config
print("\nFinal θ for α=0.1, 100 iterations:")
theta_best_gd = gradient_descent(X_train_sc, y_train, 0.1, 100)
print(f"  Intercept: {theta_best_gd[0]:.4f}")
for name, c in zip(feature_names, theta_best_gd[1:]):
    print(f"  {name:20s}: {c:.4f}")



     α | Iters |    Train MSE   Train R² |     Test MSE    Test R²
---------------------------------------------------------------------------
  0.01 |    10 |  235727.7698    -1.0474 |  280568.7055    -0.6828
  0.01 |    50 |   69720.4989     0.3945 |   97049.5408     0.4179
  0.01 |   100 |   36820.3499     0.6802 |   63333.0351     0.6201
  0.10 |    10 |   35105.1019     0.6951 |   61630.4335     0.6304
  0.10 |    50 |   31497.2612     0.7264 |   57722.4753     0.6538
  0.10 |   100 |   31486.4318     0.7265 |   57638.9572     0.6543
  0.50 |    10 |     DIVERGED            |     DIVERGED           
  0.50 |    50 |     DIVERGED            |     DIVERGED           
  0.50 |   100 |     DIVERGED            |     DIVERGED           

Final θ for α=0.1, 100 iterations:
  Intercept: 520.4148
  bedrooms            : -12.5191
  bathrooms           : 18.4242
  sqft_living         : 56.7696
  sqft_lot            : 10.2707
  floors              : 8.0610
  waterfront          : 63.6981
  v

In [21]:
# PROBLEM 6: Ridge Regularization

# Part 2: Ridge with gradient descent
def gradient_descent_ridge(X, y, alpha, n_iters, lam):
    N, d = X.shape
    X_b = np.hstack([np.ones((N, 1)), X])
    theta = np.zeros(d + 1)

    for _ in range(n_iters):
        predictions = X_b @ theta
        errors = predictions - y
        gradient = (2.0 / N) * (X_b.T @ errors)
        # Regularization gradient (don't penalize bias)
        reg_grad = 2 * lam * theta
        reg_grad[0] = 0
        gradient += (1.0 / N) * reg_grad
        theta = theta - alpha * gradient

    return theta

# Part 3: Simulation
print("\n--- Part 3: Simulated data with Ridge ---")
np.random.seed(42)
N_sim = 1000
X_sim = np.random.uniform(-2, 2, size=(N_sim, 1))
e_sim = np.random.normal(0, 2, size=N_sim)
y_sim = 1 + 2 * X_sim.flatten() + e_sim

# Closed-form ridge
def ridge_closed_form(X, y, lam):
    N = X.shape[0]
    X_b = np.hstack([np.ones((N, 1)), X])
    d = X_b.shape[1]
    I_reg = np.eye(d)
    I_reg[0, 0] = 0  # don't penalize bias
    theta = np.linalg.inv(X_b.T @ X_b + lam * I_reg) @ X_b.T @ y
    return theta

# Standard linear regression (lambda=0)
theta_ols = ridge_closed_form(X_sim, y_sim, 0)
y_ols = closed_form_predict(X_sim, theta_ols)
print(f"\nOLS: intercept={theta_ols[0]:.4f}, slope={theta_ols[1]:.4f}, "
      f"MSE={mean_squared_error(y_sim, y_ols):.4f}, R²={r2_score(y_sim, y_ols):.4f}")

print(f"\n{'λ':>8} | {'Intercept':>10} {'Slope':>10} | {'MSE':>10} {'R²':>10}")
print("-" * 58)

for lam in [1, 10, 100, 1000, 10000]:
    theta_r = ridge_closed_form(X_sim, y_sim, lam)
    y_r = closed_form_predict(X_sim, theta_r)
    mse_r = mean_squared_error(y_sim, y_r)
    r2_r = r2_score(y_sim, y_r)
    print(f"{lam:8d} | {theta_r[0]:10.4f} {theta_r[1]:10.4f} | {mse_r:10.4f} {r2_r:10.4f}")



--- Part 3: Simulated data with Ridge ---

OLS: intercept=1.1948, slope=1.9226, MSE=3.8999, R²=0.5639

       λ |  Intercept      Slope |        MSE         R²
----------------------------------------------------------
       1 |     1.1947     1.9212 |     3.8999     0.5639
      10 |     1.1942     1.9086 |     3.9001     0.5639
     100 |     1.1897     1.7913 |     3.9234     0.5613
    1000 |     1.1631     1.1094 |     4.8021     0.4630
   10000 |     1.1288     0.2308 |     7.8044     0.1273
